# Алгоритмы индексации

###  Метрика качества: Recall@k

Как понять, насколько хорош индекс? Для этого используют метрику Recall@k.

Recall@k показывает, какую долю из k истинно ближайших соседей нашёл алгоритм.

Пример: если точный поиск нашёл бы векторы [1, 5, 7, 9, 12], а приближённый нашёл [1, 5, 8, 9, 15], то Recall@5 = 3/5 = 60% (совпали 1, 5, 9).



In [1]:
# Измерение Recall@k
import numpy as np
import faiss

def compute_recall(true_neighbors: np.ndarray, found_neighbors: np.ndarray, k: int) -> float:
    """
    Вычисление Recall@k.
    
    Args:
        true_neighbors: истинные ближайшие соседи (из точного поиска)
        found_neighbors: найденные соседи (из приближённого поиска)
        k: количество соседей для проверки
    """
    recall_sum = 0
    for i in range(len(true_neighbors)):
        true_set = set(true_neighbors[i, :k])
        found_set = set(found_neighbors[i, :k])
        recall_sum += len(true_set & found_set) / k
    return recall_sum / len(true_neighbors)

# Пример использования
dimension = 128
nb = 10000
queries_count = 100
k = 10

# Генерация данных
np.random.seed(42)
vectors = np.random.random((nb, dimension)).astype('float32')
queries = np.random.random((queries_count, dimension)).astype('float32')

# Точный поиск (эталон)
index_flat = faiss.IndexFlatL2(dimension)
index_flat.add(vectors)
D_true, I_true = index_flat.search(queries, k)

# Приближённый поиск (проверяем)
index_approx = faiss.IndexFlatL2(dimension)  # пока тот же для примера
index_approx.add(vectors)
D_found, I_found = index_approx.search(queries, k)

# Вычисление Recall
recall = compute_recall(I_true, I_found, k)
print(f"Recall@{k}: {recall:.2%}")
# Вывод: Recall@10: 100.00% (так как оба индекса одинаковые)

Recall@10: 100.00%


## Уровень 0: IndexFlat — эталон точности

IndexFlatL2 (или IndexFlatIP для косинусного расстояния) - это полный перебор всех векторов. Он гарантирует 100% Recall, но
медленный.

Когда использовать:
- Датасет маленький (<10,000 векторов)
- Нужна абсолютная точность
- Как эталон для измерения Recall других индексов


In [3]:
import faiss
import numpy as np

dimension = 128
nb = 10000
vectors = np.random.random((nb, dimension)).astype('float32')

# Создание индекса
index = faiss.IndexFlatL2(dimension)
index.add(vectors)

# Поиск
query = np.random.random((1, dimension)).astype('float32')
D, I = index.search(query, k=10)

print(f"Найдено {len(I[0])} соседей")
print(f"Размер в памяти: {nb * dimension * 4 / (1024**2):.1f} МБ")
# Для 10K векторов 128D: ~5 МБ


Найдено 10 соседей
Размер в памяти: 4.9 МБ


#### Проблема IndexFlat

При 1 миллионе векторов:
- Память: ~500 МБ (ещё нормально)
- Скорость: ~10-50 мс на запрос (уже медленно)

При 10 миллионах:
- Память: ~5 ГБ (критично)
- Скорость: ~100-500 мс (неприемлемо для production)

Вывод: IndexFlat — отличный baseline, но для production нужно что-то быстрее.- 

## Уровень 1: IVF - ускорение через кластеризацию

IVF (Inverted File) решает проблему скорости, разделяя векторы на кластеры.

Аналогия

Представьте библиотеку, где книги разложены по тематическим залам (кластерам). Вместо того чтобы искать нужную книгу по всей
библиотеке, вы:

1. Определяете, в каком зале искать (ближайшие кластеры)
2. Ищете только в этих залах, игнорируя остальные

Как работает IVF

1. Обучение: Векторы группируются в кластеры (обычно k-means)
2. Индексация: Каждый вектор приписывается к ближайшему кластеру
3. Поиск: Находим ближайшие кластеры к запросу, ищем только в них

Ключевые параметры:
- nlist: количество кластеров
    - Больше → точнее разбиение, но медленнее обучение
    - Меньше → быстрее, но грубее
    - Типичная эвристика: Vn (для 100K векторов = 316 кластеров)
- nprobe: в скольких кластерах искать
    - 1 → очень быстро, но низкий recall
    - 10-20 → хороший баланс
    - nlist → превращается в IndexFlat

In [ ]:
import faiss
import numpy as np

dimension = 128
nb = 100000
vectors = np.random.random((nb, dimension)).astype('float32')
queries = np.random.random((100, dimension)).astype('float32')

# Создание IVF индекса
nlist = 100  # количество кластеров
quantizer = faiss.IndexFlatL2(dimension)
index_ivf = faiss.IndexIVFFlat(quantizer, dimension, nlist)

# Обучение (обязательно для IVF!)
print("Обучение индекса...")
index_ivf.train(vectors)

# Добавление векторов
index_ivf.add(vectors)

# Эталонный поиск для сравнения
index_flat = faiss.IndexFlatL2(dimension)
index_flat.add(vectors)
D_true, I_true = index_flat.search(queries, 10)

# Тест разных nprobe
for nprobe in [1, 5, 10, 20]:
    index_ivf.nprobe = nprobe
    D_found, I_found = index_ivf.search(queries, 10)
    recall = compute_recall(I_true, I_found, 10)
    print(f"nprobe={nprobe}: Recall@10 = {recall:.1%}")

Обучение индекса...
nprobe=1: Recall@10 = 5.4%
nprobe=5: Recall@10 = 20.8%
nprobe=10: Recall@10 = 34.1%
nprobe=20: Recall@10 = 52.3%


Что улучшилось по сравнению с IndexFlat
- Скорость: В 5-10 раз быстрее при nprobe=10
- Память: Примерно такая же (векторы всё ещё хранятся полностью)
- Recall: 90-98% при правильной настройке

Недостаток IVF

Память всё ещё проблема для очень больших датасетов. При 100 миллионах векторов (128D) нужно ~50 ГБ RAM — это уже дорого.

Вывод: IVF отлично ускоряет поиск, но память остаётся проблемой для больших данных.

## Уровень 2: HNSW — графовый индекс

HNSW (Hierarchical Navigable Small World) - это графовый алгоритм, который даёт лучшее соотношение скорости и точности среди
всех методов.

Аналогия

Представьте карту городов с дорогами разного уровня:
- Автострады (верхний уровень) — соединяют крупные города, быстро покрывают большие расстояния
- Региональные дороги (средний уровень) — соединяют города внутри региона
- Улицы (нижний уровень) - соединяют все точки детально

Поиск начинается "сверху" (по автострадам), быстро приближаясь к цели, затем спускается на детальные уровни для точного
попадания.

Как работает HNSW

HNSW строит многоуровневый граф, где:

1. Каждый вектор — это узел
2. Узлы соединены рёбрами с М ближайшими соседями
3. Верхние уровни разрежены (мало узлов), нижние плотные (все узлы)
4. Поиск начинается сверху и спускается вниз

Ключевые параметры

- М: количество связей каждого узла (типично 16-32)
  - Больше → точнее, но больше памяти и медленнее построение
  - Меньше → быстрее построение, меньше памяти, хуже recall

- efConstruction: качество построения графа (типично 100-400)
  - Больше → лучше граф, медленнее построение
  - Это параметр только для построения, потом не меняется

- efSearch: качество поиска (типично 50-200)
  - Больше → точнее поиск, медленнее
  - Можно менять после построения!

In [6]:
import faiss
import numpy as np

dimension = 128
nb = 100000
vectors = np.random.random((nb, dimension)).astype('float32')
queries = np.random.random((100, dimension)).astype('float32')

# Создание HNSW индекса
M = 32
index_hnsw = faiss.IndexHNSWFlat(dimension, M)

# Настройка параметров
index_hnsw.hnsw.efConstruction = 200  # качество построения
print("Построение индекса...")
index_hnsw.add(vectors)

# Эталонный поиск
index_flat = faiss.IndexFlatL2(dimension)
index_flat.add(vectors)
D_true, I_true = index_flat.search(queries, 10)

# Тест разных efSearch
for ef_search in [16, 32, 64, 128]:
    index_hnsw.hnsw.efSearch = ef_search
    D_found, I_found = index_hnsw.search(queries, 10)
    recall = compute_recall(I_true, I_found, 10)
    print(f"efSearch={ef_search}: Recall@10 = {recall:.1%}")

Построение индекса...
efSearch=16: Recall@10 = 25.6%
efSearch=32: Recall@10 = 38.4%
efSearch=64: Recall@10 = 59.0%
efSearch=128: Recall@10 = 76.0%


Что улучшилось по сравнению с IVF

- Скорость: Быстрее IVF при том же recall (особенно на больших данных)
- Recall: Стабильно 95-99% при разумных настройках
- Гибкость: efSearch можно менять после построения

Критический недостаток HNSW

Память растёт быстрее, чем сами векторы!

Для каждого вектора хранятся:

- Сам вектор (128D × 4 байта = 512 байт)
- Связи графа (M × 4 байта × уровни « 128-256 байт)

Итого ~700-800 байт на вектор вместо 512.

При 100 миллионах векторов:

- IndexFlat: ~50 ГБ
- HNSW: ~70-80 ГБ

Это может стать блокирующим фактором.

Вывод: HNSW — оптимальный выбор для большинства задач, но при очень больших данных упирается в память.

## Уровень 3: IVF-PQ - сжатие для больших данных

Когда данных десятки миллионов и память критична, используют Product Quantization (PQ) - метод сжатия векторов.

Как работает PQ

PQ разбивает вектор на части и квантует каждую часть:

1. Вектор 128D делится на 16 подвекторов по 8D
2. Каждый подвектор заменяется индексом ближайшего центроида (256 вариантов)
3. Вместо 512 байт (128 × float32) храним 16 байт (16 × uint8)

Сжатие: 32х!

Критическое предупреждение о PQ

A PQ не бесплатен:
- Ломает локальную геометрию пространства
- Плохо работает на коррелированных embeddings
- Может резко снизить recall, особенно без IVF
- PQ почти никогда не используется в одиночку - только с IVF

In [8]:
import faiss
import numpy as np

dimension = 128
nb = 100000
vectors = np.random.random((nb, dimension)).astype('float32')
queries = np.random.random((100, dimension)).astype('float32')

# Параметры
nlist = 100  # кластеры IVF
m = 16  # количество подпространств PQ (должно делить dimension)
nbits = 8  # бит на подпространство (обычно 8)

# Создание IVF-PQ индекса
quantizer = faiss.IndexFlatL2(dimension)
index_ivfpq = faiss.IndexIVFPQ(quantizer, dimension, nlist, m, nbits)

# Обучение
print("Обучение индекса (IVF + PQ)...")
index_ivfpq.train(vectors)
index_ivfpq.add(vectors)

# Эталон
index_flat = faiss.IndexFlatL2(dimension)
index_flat.add(vectors)
D_true, I_true = index_flat.search(queries, 10)

# Тест
index_ivfpq.nprobe = 10
D_found, I_found = index_ivfpq.search(queries, 10)
recall = compute_recall(I_true, I_found, 10)

# Сравнение размеров
original_size = nb * dimension * 4
compressed_size = nb * m * (nbits // 8)

print(f"\nRecall@10: {recall:.1%}")
print(f"Оригинальный размер: {original_size / (1024**2):.1f} МБ")
print(f"Сжатый размер: {compressed_size / (1024**2):.1f} МБ")
print(f"Сжатие: {original_size / compressed_size:.1f}x")

Обучение индекса (IVF + PQ)...

Recall@10: 13.8%
Оригинальный размер: 48.8 МБ
Сжатый размер: 1.5 МБ
Сжатие: 32.0x


Что улучшилось по сравнению с HNSW

• Память: В 30-50 раз меньше!
• Масштабируемость: Можно работать с миллиардами векторов

Что ухудшилось

• Recall: Обычно 85-95% (ниже чем HNSW)
• Качество: На некоторых данных может сильно просесть

Вывод: IVF-PQ — это когда память реально критична. В остальных случаях лучше HNSW.

### Настройка HNSW в Qdrant


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, HnswConfigDiff

client = QdrantClient(url="http://localhost:6333")

# Настройка HNSW при создании коллекции
client.create_collection(
    collection_name="optimized_collection",
    vectors_config=VectorParams(
        size=128,
        distance=Distance.COSINE,
        hnsw_config=HnswConfigDiff(
            m=32,                    # количество связей (16-64)
            ef_construct=200,        # качество построения (100-500)
            full_scan_threshold=10000  # порог переключения на полный скан
        )
    )
)

# Поиск с настройкой efSearch (search() удалён, используем query_points())
from qdrant_client.models import SearchParams

results = client.query_points(
    collection_name="optimized_collection",
    query=[0.1, 0.2, ...],
    limit=10,
    search_params=SearchParams(
        hnsw_ef=128  # efSearch для этого запроса
    )
)

Итоги

Прошли путь от простого к оптимальному:
- IndexFlat - эталон точности, но медленный
- IVF - ускорение через кластеризацию, экономит время
- HNSW — оптимальный выбор для большинства задач (95-99% recall, быстро)
- IVF-PQ — когда память критична (30х сжатие, но recall 85-95%)